In [7]:
import os
import json
from statistics import mean

project_root = r"C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process"
base_folder = fr"{project_root}\data\Documents_Annotés\llm"



prefix = "history_"



In [4]:
def show_failure_ratios(experiments):
    experiment_results = {}

    for exp in experiments:
        folder = os.path.join(base_folder, exp)

        total_chunks = 0
        failed_chunks = 0

        for filename in os.listdir(folder):
            if filename.startswith(prefix) and filename.endswith(".json"):
                filepath = os.path.join(folder, filename)

                with open(filepath, "r", encoding="utf-8") as f:
                    data = json.load(f)

                if not data:
                    continue

                # iterate over chunks
                for chunk in data:
                    total_chunks += 1

                    # normalize chunk structure
                    if isinstance(chunk, list):
                        iterable = chunk
                    else:
                        iterable = [chunk]

                    # check if ANY item in chunk failed
                    chunk_failed = any(
                        "Success" != item.get("status", "")
                        for item in iterable
                    )

                    if chunk_failed:
                        failed_chunks += 1

        ratio = failed_chunks / total_chunks if total_chunks > 0 else 0

        experiment_results[exp] = {
            "total_chunks": total_chunks,
            "failed_chunks": failed_chunks,
            "failure_ratio": ratio
        }

    # ---- Global stats ----
    all_ratios = [v["failure_ratio"] for v in experiment_results.values()]
    global_mean_ratio = mean(all_ratios) if all_ratios else 0

    total_failed_chunks = sum(v["failed_chunks"] for v in experiment_results.values())
    total_chunks_all = sum(v["total_chunks"] for v in experiment_results.values())
    global_ratio = total_failed_chunks / total_chunks_all if total_chunks_all > 0 else 0

    # ---- Print results ----
    for exp, stats in experiment_results.items():
        print(f"{exp}")
        print(f"  chunks: {stats['total_chunks']}")
        print(f"  failed chunks: {stats['failed_chunks']}")
        print(f"  failure ratio: {stats['failure_ratio']:.4f}")
        print()

    print(f"Total chunks: {total_chunks_all}. Failed chunks: {total_failed_chunks}.")
    print(f"GLOBAL FAILURE RATIO (ALL CHUNKS): {global_ratio*100:.2f}%")

In [10]:
from pathlib import Path
from bs4 import BeautifulSoup

GOOD_LABELS      = {"legislation", "decision", "secondary sources"}
STRUCTURAL_LABELS = {"title", "citation", "source", "authors"}

BASE_FOLDER = Path(r"C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm")


def _pick_html_files(folder: Path) -> list[Path]:
    """Return v1.3.html files if any exist, else fall back to v1.0.html."""
    v13 = sorted(folder.glob("*v1.3.html"))
    if v13:
        print(f"[INFO] Found {len(v13)} v1.3 HTML files in: {folder}")
        return v13
    return sorted(folder.glob("*v1.0.html"))


def _has_auto_label_ancestor(tag) -> bool:
    """Return True if any ancestor of this tag is also an <auto_label>."""
    for parent in tag.parents:
        if getattr(parent, "name", None) == "auto_label":
            return True
    return False


def show_not_nested_label_ratio(
    experiments: list[str],
    base_path: Path = BASE_FOLDER,
) -> None:
    total_good    = 0
    total_failure = 0
    per_experiment = {}

    for exp_name in experiments:
        folder     = base_path / exp_name
        html_files = _pick_html_files(folder)

        if not html_files:
            print(f"[WARN] No target HTML files found in: {folder}")
            per_experiment[exp_name] = {"good": 0, "failure": 0, "files": 0}
            continue

        exp_good    = 0
        exp_failure = 0

        for html_file in html_files:
            soup = BeautifulSoup(html_file.read_text(encoding="utf-8"), "html.parser")

            for tag in soup.find_all("auto_label"):
                labelname = (tag.get("labelname") or "").strip().lower()

                if labelname in GOOD_LABELS:
                    exp_good += 1

                elif labelname in STRUCTURAL_LABELS:
                    if _has_auto_label_ancestor(tag):
                        exp_good += 1      # nested inside another auto_label → fine
                    else:
                        exp_failure += 1   # floating at root level → failure

        total_good    += exp_good
        total_failure += exp_failure
        per_experiment[exp_name] = {
            "good":    exp_good,
            "failure": exp_failure,
            "files":   len(html_files),
        }

    # ── Report ───────────────────────────────────────────────────────────────
    col = 58
    print(f"\n{'Experiment':<{col}} {'Files':>5}  {'Good':>6}  {'Fail':>6}  {'Good%':>7}")
    print("─" * (col + 30))

    for exp_name, stats in per_experiment.items():
        g, f   = stats["good"], stats["failure"]
        total  = g + f
        ratio  = f"{g / total:.1%}" if total else "  n/a"
        print(f"{exp_name:<{col}} {stats['files']:>5}  {g:>6}  {f:>6}  {ratio:>7}")

    print("─" * (col + 30))
    grand_total = total_good + total_failure
    grand_ratio = f"{total_good / grand_total:.1%}" if grand_total else "n/a"
    print(f"{'TOTAL':<{col}} {'':>5}  {total_good:>6}  {total_failure:>6}  {grand_ratio:>7}\n")

In [8]:
experiments = [
    "PARACHUNKER_ALLINONE_p2_c500_fsselected-30_mgpt-5.2",
    "PARACHUNKER_ALLINONE_p2_c500_fsrandom-30_mgpt-5.2",
    "PARACHUNKER_ALLINONE_p2_c500_fsrandom-15_mgpt-5.2",
    "PARACHUNKER_ALLINONE_p2_c500_fspattern-30_mgpt-5.2",
    "PARACHUNKER_ALLINONE_p2_c500_fspattern-15_mgpt-5.2",
    "PARACHUNKER_ALLINONE_p2_c500_fspattern-5-25_mgpt-5.2",
    "PARACHUNKER_ALLINONE_p2_c500_fspattern-5_mgpt-5.2",
]

show_failure_ratios(experiments)
show_not_nested_label_ratio(experiments)

PARACHUNKER_ALLINONE_p2_c500_fsselected-30_mgpt-5.2
  chunks: 486
  failed chunks: 10
  failure ratio: 0.0206

PARACHUNKER_ALLINONE_p2_c500_fsrandom-30_mgpt-5.2
  chunks: 772
  failed chunks: 15
  failure ratio: 0.0194

PARACHUNKER_ALLINONE_p2_c500_fsrandom-15_mgpt-5.2
  chunks: 525
  failed chunks: 15
  failure ratio: 0.0286

PARACHUNKER_ALLINONE_p2_c500_fspattern-30_mgpt-5.2
  chunks: 772
  failed chunks: 15
  failure ratio: 0.0194

PARACHUNKER_ALLINONE_p2_c500_fspattern-15_mgpt-5.2
  chunks: 772
  failed chunks: 16
  failure ratio: 0.0207

PARACHUNKER_ALLINONE_p2_c500_fspattern-5-25_mgpt-5.2
  chunks: 772
  failed chunks: 16
  failure ratio: 0.0207

PARACHUNKER_ALLINONE_p2_c500_fspattern-5_mgpt-5.2
  chunks: 772
  failed chunks: 19
  failure ratio: 0.0246

Total chunks: 4871. Failed chunks: 106.
GLOBAL FAILURE RATIO (ALL CHUNKS): 2.18%

Experiment                                                 Files    Good    Fail    Good%
──────────────────────────────────────────────────────────

In [11]:
experiments = [
    "TEST_CONTEXT_p2_c500_fsselected-30_mgpt-5.2",
    "PARACHUNKER_CONTEXT_p2_c500_fsselected-30_mgpt-5.2",
    "p2_c500_fsselected-30_mgpt-5.2",
    "p2_c500_fsselected-0_mgpt-5.2"
]

show_failure_ratios(experiments)
show_not_nested_label_ratio(experiments)

TEST_CONTEXT_p2_c500_fsselected-30_mgpt-5.2
  chunks: 5441
  failed chunks: 2
  failure ratio: 0.0004

PARACHUNKER_CONTEXT_p2_c500_fsselected-30_mgpt-5.2
  chunks: 7924
  failed chunks: 2
  failure ratio: 0.0003

p2_c500_fsselected-30_mgpt-5.2
  chunks: 488
  failed chunks: 3
  failure ratio: 0.0061

p2_c500_fsselected-0_mgpt-5.2
  chunks: 3218
  failed chunks: 6
  failure ratio: 0.0019

Total chunks: 17071. Failed chunks: 13.
GLOBAL FAILURE RATIO (ALL CHUNKS): 0.08%
[INFO] Found 7 v1.3 HTML files in: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\TEST_CONTEXT_p2_c500_fsselected-30_mgpt-5.2
[INFO] Found 11 v1.3 HTML files in: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\PARACHUNKER_CONTEXT_p2_c500_fsselected-30_mgpt-5.2
[INFO] Found 8 v1.3 HTML files in: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\p2_c500_fsselected-30_mgpt-5.2
[INFO] Found 7 v1.3 

In [28]:
experiments = [
    "F_PARACHUNKER_ALLINONE_p2_c500_fspattern-5_mgpt-5.2",
    "F_PARACHUNKER_ALLINONE_p2_c500_fspattern-5-25_mgpt-5.2"
]

show_failure_ratios(experiments)

F_PARACHUNKER_ALLINONE_p2_c500_fspattern-5_mgpt-5.2
  chunks: 772
  failed chunks: 3
  failure ratio: 0.0039

F_PARACHUNKER_ALLINONE_p2_c500_fspattern-5-25_mgpt-5.2
  chunks: 803
  failed chunks: 4
  failure ratio: 0.0050

Total chunks: 1575. Failed chunks: 7.
GLOBAL FAILURE RATIO (ALL CHUNKS): 0.44%


In [12]:
experiments = [
        "F_PARACHUNKER_ALLINONE_p2_c500_fspattern-5-25_mgpt-5.2"
]

show_not_nested_label_ratio(experiments)


Experiment                                                 Files    Good    Fail    Good%
────────────────────────────────────────────────────────────────────────────────────────
F_PARACHUNKER_ALLINONE_p2_c500_fspattern-5-25_mgpt-5.2        11    9405      19    99.8%
────────────────────────────────────────────────────────────────────────────────────────
TOTAL                                                               9405      19    99.8%

